# 🧠 Deep Learning — Complete Expert Cheat Sheet
### PyTorch + TensorFlow/Keras Side-by-Side | Every Concept from Zero to Expert

---

## 📚 Table of Contents

| # | Topic |
|---|-------|
| 1 | Setup & Imports |
| 2 | Tensors — Creation, Ops, Indexing |
| 3 | Autograd & Backpropagation |
| 4 | Neural Network Building Blocks |
| 5 | Activation Functions |
| 6 | Loss Functions |
| 7 | Optimizers |
| 8 | Training Loop (Full Pipeline) |
| 9 | Datasets & DataLoaders |
| 10 | Convolutional Neural Networks (CNNs) |
| 11 | Recurrent Neural Networks (RNNs, LSTM, GRU) |
| 12 | Transformers & Attention |
| 13 | Transfer Learning & Fine-tuning |
| 14 | Regularization Techniques |
| 15 | Batch Normalization & Layer Norm |
| 16 | Learning Rate Scheduling |
| 17 | Model Saving & Loading |
| 18 | GPU Acceleration |
| 19 | Generative Models (VAE, GAN) |
| 20 | Object Detection & Segmentation |
| 21 | Natural Language Processing (NLP) |
| 22 | Embeddings |
| 23 | Model Evaluation & Metrics |
| 24 | Hyperparameter Tuning |
| 25 | Mixed Precision Training |
| 26 | Distributed Training |
| 27 | ONNX & Model Deployment |
| 28 | Custom Layers & Modules |
| 29 | Debugging & Visualization |
| 30 | Best Practices & Tips |

---
> **How to use this notebook:** Each section contains concept explanations, PyTorch code, TensorFlow/Keras code, and pro tips. Run each cell to see outputs. Side-by-side comparisons help you master both frameworks simultaneously.

---
## 1. 🛠️ Setup & Imports

In [ ]:
# ── Install (run once) ──────────────────────────────────────────────────────
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
# !pip install tensorflow keras
# !pip install numpy matplotlib scikit-learn pandas torchsummary

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PYTORCH IMPORTS
# ══════════════════════════════════════════════════════════════════════════════
import torch
import torch.nn as nn                        # Neural network modules
import torch.nn.functional as F              # Stateless functions (relu, softmax…)
import torch.optim as optim                  # Optimizers
from torch.utils.data import Dataset, DataLoader, random_split, TensorDataset
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models          # Pretrained models
from torch.cuda.amp import GradScaler, autocast  # Mixed precision

# ══════════════════════════════════════════════════════════════════════════════
#  TENSORFLOW / KERAS IMPORTS
# ══════════════════════════════════════════════════════════════════════════════
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models as keras_models, optimizers as keras_optimizers
from tensorflow.keras import losses as keras_losses, metrics as keras_metrics
from tensorflow.keras.callbacks import (ModelCheckpoint, EarlyStopping,
                                         ReduceLROnPlateau, TensorBoard)
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# ══════════════════════════════════════════════════════════════════════════════
#  GENERAL
# ══════════════════════════════════════════════════════════════════════════════
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings; warnings.filterwarnings('ignore')

# ── Reproducibility ─────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── Device ──────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch version : {torch.__version__}")
print(f"TensorFlow version: {tf.__version__}")
print(f"PyTorch device  : {device}")
print(f"GPUs available  : {tf.config.list_physical_devices('GPU')}")

---
## 2. 🔢 Tensors — Creation, Operations, Indexing

**Tensors** are the fundamental data structure of deep learning — multi-dimensional arrays with automatic differentiation support.

| Concept | PyTorch | TensorFlow |
|---------|---------|------------|
| Base class | `torch.Tensor` | `tf.Tensor` |
| Mutable variable | `torch.Tensor` (with grad) | `tf.Variable` |
| NumPy bridge | `.numpy()` / `torch.from_numpy()` | `.numpy()` / `tf.constant(arr)` |

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  TENSOR CREATION
# ══════════════════════════════════════════════════════════════════════════════

# ── PyTorch ──────────────────────────────────────────────────────────────────
pt_scalar   = torch.tensor(3.14)                      # 0-D scalar
pt_vector   = torch.tensor([1, 2, 3], dtype=torch.float32)  # 1-D
pt_matrix   = torch.tensor([[1,2],[3,4]], dtype=torch.float64)  # 2-D
pt_zeros    = torch.zeros(3, 4)                       # all zeros
pt_ones     = torch.ones(2, 3, 4)                     # all ones
pt_rand     = torch.rand(3, 3)                        # uniform [0,1)
pt_randn    = torch.randn(3, 3)                       # standard normal
pt_arange   = torch.arange(0, 10, 2)                  # [0,2,4,6,8]
pt_linspace = torch.linspace(0, 1, 5)                 # [0,.25,.5,.75,1]
pt_eye      = torch.eye(3)                            # identity matrix
pt_full     = torch.full((2, 3), fill_value=7.0)      # filled with 7
pt_like     = torch.zeros_like(pt_rand)               # same shape as rand

# ── TensorFlow ───────────────────────────────────────────────────────────────
tf_scalar   = tf.constant(3.14)
tf_vector   = tf.constant([1, 2, 3], dtype=tf.float32)
tf_matrix   = tf.constant([[1,2],[3,4]], dtype=tf.float64)
tf_zeros    = tf.zeros((3, 4))
tf_ones     = tf.ones((2, 3, 4))
tf_rand     = tf.random.uniform((3, 3))
tf_randn    = tf.random.normal((3, 3))
tf_range    = tf.range(0, 10, 2)
tf_linspace = tf.linspace(0.0, 1.0, 5)
tf_eye      = tf.eye(3)
tf_full     = tf.fill((2, 3), 7.0)
tf_like     = tf.zeros_like(tf_rand)

# ── Mutable Variables (needed for trainable params) ──────────────────────────
pt_var = torch.tensor([1.0, 2.0], requires_grad=True)  # PyTorch trainable
tf_var = tf.Variable([1.0, 2.0])                        # TF trainable

print("PT rand shape:", pt_rand.shape, "| dtype:", pt_rand.dtype)
print("TF rand shape:", tf_rand.shape, "| dtype:", tf_rand.dtype)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  TENSOR OPERATIONS
# ══════════════════════════════════════════════════════════════════════════════

a = torch.tensor([[1.,2.],[3.,4.]])
b = torch.tensor([[5.,6.],[7.,8.]])

# ── Arithmetic ───────────────────────────────────────────────────────────────
print("--- PyTorch ---")
print("Add        :", a + b)             # element-wise add
print("Subtract   :", a - b)
print("Multiply   :", a * b)             # element-wise (Hadamard)
print("MatMul     :", a @ b)             # matrix multiply  (same as torch.matmul)
print("Power      :", a ** 2)
print("Sqrt       :", torch.sqrt(a))
print("Exp        :", torch.exp(a))
print("Log        :", torch.log(a))
print("Sum        :", a.sum(), "| row sums:", a.sum(dim=1))
print("Mean       :", a.mean())
print("Max        :", a.max(), "| argmax:", a.argmax())
print("Transpose  :", a.T)              # or a.transpose(0,1)
print("Reshape    :", a.reshape(1,4))   # or a.view(1,4)
print("Squeeze    :", torch.unsqueeze(a,0).shape, "->squeeze->", torch.squeeze(torch.unsqueeze(a,0)).shape)
print("Flatten    :", torch.flatten(a))
print("Concat dim0:", torch.cat([a,b], dim=0).shape)  # (4,2)
print("Stack      :", torch.stack([a,b], dim=0).shape) # (2,2,2)
print("Clamp      :", torch.clamp(a, min=2, max=3))

# ── TensorFlow same ops ───────────────────────────────────────────────────────
ta = tf.constant([[1.,2.],[3.,4.]])
tb = tf.constant([[5.,6.],[7.,8.]])
print("\n--- TensorFlow ---")
print("Add        :", ta + tb)
print("MatMul     :", tf.linalg.matmul(ta, tb))
print("Reshape    :", tf.reshape(ta, (1,4)))
print("Concat dim0:", tf.concat([ta,tb], axis=0).shape)
print("ReduceSum  :", tf.reduce_sum(ta))
print("Transpose  :", tf.transpose(ta))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  INDEXING, SLICING, ADVANCED INDEXING
# ══════════════════════════════════════════════════════════════════════════════
x = torch.arange(1, 13).reshape(3, 4).float()
print("Tensor x:\n", x)
print("x[0]         :", x[0])           # first row
print("x[:, 1]      :", x[:, 1])        # second column
print("x[0:2, 1:3]  :\n", x[0:2, 1:3]) # slice
print("x[[0,2], :]  :\n", x[[0,2], :]) # fancy indexing
print("x > 6        :\n", x > 6)        # boolean mask
print("x[x>6]       :", x[x > 6])      # masked select

# ── gather / scatter ─────────────────────────────────────────────────────────
idx = torch.tensor([[0,1,2],[1,2,3]])
print("gather:", torch.gather(x, 1, idx))  # gather cols per row

# ── TensorFlow indexing ───────────────────────────────────────────────────────
tx = tf.cast(tf.reshape(tf.range(1,13),(3,4)), tf.float32)
print("\nTF x[0]    :", tx[0].numpy())
print("TF x[:,1]  :", tx[:,1].numpy())
print("TF boolean :", tf.boolean_mask(tx, tx > 6).numpy())

---
## 3. 🔄 Autograd & Backpropagation

**Autograd** automatically computes gradients via the chain rule. Every tensor operation is recorded in a **computation graph**.

```
Forward pass:  x → [op1] → [op2] → loss
Backward pass: loss.backward() → dL/dx computed automatically
```

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PYTORCH AUTOGRAD
# ══════════════════════════════════════════════════════════════════════════════

# ── Basic gradient computation ────────────────────────────────────────────────
x = torch.tensor(3.0, requires_grad=True)
y = x ** 3 + 2*x + 1          # y = x³ + 2x + 1
y.backward()                   # dy/dx = 3x² + 2 = 3(9)+2 = 29
print(f"x = {x.item()}, y = {y.item()}, dy/dx = {x.grad.item()}")  # 29.0

# ── Vector gradients (Jacobian-vector product) ────────────────────────────────
x = torch.tensor([1., 2., 3.], requires_grad=True)
y = (x ** 2).sum()             # scalar loss
y.backward()
print("Grad of sum(x²):", x.grad)  # [2, 4, 6]

# ── Clearing gradients (MUST do before each backward) ────────────────────────
x.grad.zero_()    # in-place zero; optimizer.zero_grad() does this for all params

# ── retain_graph: needed for multiple backward passes ────────────────────────
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2
y.backward(retain_graph=True)   # keep graph for second backward
y.backward()                    # second call
print("Grad after 2x backward:", x.grad)  # 4 + 4 = 8 (accumulated!)

# ── Stopping gradient flow ────────────────────────────────────────────────────
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2
z = y.detach()         # z is a new tensor, no grad
print("z.requires_grad:", z.requires_grad)  # False

with torch.no_grad():  # context manager — no grad computation (faster inference)
    result = x ** 3
print("no_grad result.requires_grad:", result.requires_grad)  # False

# ── Gradient of arbitrary function (torch.autograd.grad) ────────────────────
x = torch.tensor(3.0, requires_grad=True)
y = x ** 4
grads = torch.autograd.grad(y, x)
print("dy/dx via autograd.grad:", grads[0])  # 4x³ = 108

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  TENSORFLOW GRADIENT TAPE
# ══════════════════════════════════════════════════════════════════════════════

# ── Basic ─────────────────────────────────────────────────────────────────────
x = tf.Variable(3.0)
with tf.GradientTape() as tape:
    y = x**3 + 2*x + 1
dy_dx = tape.gradient(y, x)
print(f"x={x.numpy()}, y={y.numpy()}, dy/dx={dy_dx.numpy()}")  # 29.0

# ── Multiple gradients at once ────────────────────────────────────────────────
x = tf.Variable([1., 2., 3.])
with tf.GradientTape() as tape:
    y = tf.reduce_sum(x**2)
grad = tape.gradient(y, x)
print("Grads:", grad.numpy())  # [2, 4, 6]

# ── Persistent tape (multiple gradient calls) ────────────────────────────────
x = tf.Variable(2.0)
with tf.GradientTape(persistent=True) as tape:
    y = x**2
    z = x**3
print("dy/dx:", tape.gradient(y, x).numpy())   # 4
print("dz/dx:", tape.gradient(z, x).numpy())   # 12
del tape

# ── Watching non-variable tensors ────────────────────────────────────────────
x = tf.constant(3.0)
with tf.GradientTape() as tape:
    tape.watch(x)          # must explicitly watch tf.constant
    y = x**2
print("Grad of constant:", tape.gradient(y, x).numpy())  # 6

# ── Higher-order gradients ────────────────────────────────────────────────────
x = tf.Variable(3.0)
with tf.GradientTape() as t2:
    with tf.GradientTape() as t1:
        y = x**3
    dy_dx = t1.gradient(y, x)     # first derivative: 3x² = 27
d2y_dx2 = t2.gradient(dy_dx, x)  # second derivative: 6x = 18
print(f"d2y/dx2 at x=3: {d2y_dx2.numpy()}")  # 18

---
## 4. 🧱 Neural Network Building Blocks

Core layers that form every deep learning architecture:

| Layer | Purpose |
|-------|---------|
| `Linear` / `Dense` | Fully-connected: `y = Wx + b` |
| `Conv2d` | Spatial feature extraction |
| `BatchNorm` | Normalize activations |
| `Dropout` | Regularization |
| `Embedding` | Lookup table for tokens |
| `LSTM`/`GRU` | Sequential memory |

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PYTORCH — Building a Model (Multiple Ways)
# ══════════════════════════════════════════════════════════════════════════════

# ── Method 1: nn.Sequential (simple stacks) ───────────────────────────────────
model_seq = nn.Sequential(
    nn.Linear(784, 256),    # in_features, out_features
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, 128),
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Linear(128, 10),
)

# ── Method 2: Class-based (recommended for complex models) ────────────────────
class MLP(nn.Module):
    def __init__(self, in_dim, hidden_dims, out_dim, dropout=0.3):
        super().__init__()
        layers_list = []
        dims = [in_dim] + hidden_dims
        for i in range(len(dims)-1):
            layers_list += [nn.Linear(dims[i], dims[i+1]),
                            nn.BatchNorm1d(dims[i+1]),
                            nn.ReLU(),
                            nn.Dropout(dropout)]
        layers_list.append(nn.Linear(dims[-1], out_dim))
        self.net = nn.Sequential(*layers_list)

    def forward(self, x):
        return self.net(x)

model_mlp = MLP(784, [512, 256, 128], 10)
model_mlp = model_mlp.to(device)

# ── Inspect model ────────────────────────────────────────────────────────────
x_dummy = torch.randn(32, 784).to(device)  # batch=32, features=784
out = model_mlp(x_dummy)
print("Output shape:", out.shape)           # (32, 10)

# Count parameters
total_params = sum(p.numel() for p in model_mlp.parameters())
trainable_params = sum(p.numel() for p in model_mlp.parameters() if p.requires_grad)
print(f"Total params: {total_params:,} | Trainable: {trainable_params:,}")

# Print model architecture
print(model_mlp)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  TENSORFLOW / KERAS — Building a Model
# ══════════════════════════════════════════════════════════════════════════════

# ── Method 1: Sequential API ──────────────────────────────────────────────────
model_seq_tf = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dense(10, activation='softmax')
])

# ── Method 2: Functional API (for multi-input/output, skip connections) ───────
inputs = keras.Input(shape=(784,), name='input')
x = layers.Dense(512, activation='relu')(inputs)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(10, activation='softmax', name='output')(x)
model_func_tf = keras.Model(inputs=inputs, outputs=outputs, name='FunctionalMLP')

# ── Method 3: Subclassing (like PyTorch Module) ───────────────────────────────
class MLPKeras(keras.Model):
    def __init__(self, hidden_units, num_classes):
        super().__init__()
        self.hidden_layers = [layers.Dense(u, activation='relu') for u in hidden_units]
        self.bn_layers     = [layers.BatchNormalization() for _ in hidden_units]
        self.dropout       = layers.Dropout(0.3)
        self.output_layer  = layers.Dense(num_classes, activation='softmax')

    def call(self, x, training=False):
        for dense, bn in zip(self.hidden_layers, self.bn_layers):
            x = dense(x)
            x = bn(x, training=training)
            x = self.dropout(x, training=training)
        return self.output_layer(x)

model_tf = MLPKeras([512, 256, 128], 10)

# ── Summary ───────────────────────────────────────────────────────────────────
model_func_tf.summary()

---
## 5. ⚡ Activation Functions

Activations introduce **non-linearity**, enabling networks to learn complex patterns.

| Function | Formula | Use Case |
|----------|---------|----------|
| ReLU | `max(0,x)` | Default hidden layers |
| LeakyReLU | `max(αx,x)` | Avoids dying ReLU |
| ELU | `x if x>0 else α(eˣ-1)` | Smoother than ReLU |
| GELU | `x·Φ(x)` | Transformers (BERT, GPT) |
| Sigmoid | `1/(1+e⁻ˣ)` | Binary output |
| Tanh | `(eˣ-e⁻ˣ)/(eˣ+e⁻ˣ)` | RNNs, [-1,1] range |
| Softmax | `eˣᵢ/Σeˣⱼ` | Multi-class output |
| Swish | `x·sigmoid(x)` | EfficientNet |

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  ALL ACTIVATION FUNCTIONS + VISUALIZATION
# ══════════════════════════════════════════════════════════════════════════════
x_vals = torch.linspace(-4, 4, 200)

activations_pt = {
    'ReLU'       : F.relu(x_vals),
    'LeakyReLU'  : F.leaky_relu(x_vals, 0.1),
    'ELU'        : F.elu(x_vals),
    'GELU'       : F.gelu(x_vals),
    'Sigmoid'    : torch.sigmoid(x_vals),
    'Tanh'       : torch.tanh(x_vals),
    'Softplus'   : F.softplus(x_vals),
    'Swish'      : x_vals * torch.sigmoid(x_vals),
    'Mish'       : x_vals * torch.tanh(F.softplus(x_vals)),
    'SELU'       : F.selu(x_vals),
}

fig, axes = plt.subplots(2, 5, figsize=(20, 7))
axes = axes.flatten()
x_np = x_vals.numpy()
for ax, (name, y) in zip(axes, activations_pt.items()):
    ax.plot(x_np, y.numpy(), linewidth=2, color='steelblue')
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.axvline(0, color='gray', linewidth=0.5)
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.set_xlim(-4, 4); ax.grid(True, alpha=0.3)
plt.suptitle('Activation Functions', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

# ── PyTorch usage ─────────────────────────────────────────────────────────────
x = torch.randn(5)
print("ReLU   (F)  :", F.relu(x))           # functional
print("ReLU   (nn) :", nn.ReLU()(x))        # as module
print("Sigmoid     :", torch.sigmoid(x))
print("Softmax     :", F.softmax(x, dim=0))
print("LogSoftmax  :", F.log_softmax(x, dim=0))  # numerically stable

# ── TensorFlow usage ──────────────────────────────────────────────────────────
xt = tf.constant(x.numpy())
print("\nTF ReLU  :", tf.nn.relu(xt).numpy())
print("TF GELU  :", tf.nn.gelu(xt).numpy())
print("TF Swish :", tf.nn.swish(xt).numpy())

---
## 6. 📉 Loss Functions

Loss functions measure the **discrepancy between predictions and ground truth**.

| Loss | Task | Notes |
|------|------|-------|
| MSELoss | Regression | Sensitive to outliers |
| MAELoss | Regression | Robust to outliers |
| HuberLoss | Regression | Combines MSE + MAE |
| BCELoss | Binary classification | Requires sigmoid |
| BCEWithLogitsLoss | Binary classification | Numerically stable |
| CrossEntropyLoss | Multi-class | Includes log-softmax |
| NLLLoss | Multi-class | Use after log-softmax |
| KLDivLoss | Distribution matching | VAE, distillation |
| ContrastiveLoss | Metric learning | Siamese nets |
| TripletMarginLoss | Metric learning | Anchor-pos-neg |

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PYTORCH LOSS FUNCTIONS
# ══════════════════════════════════════════════════════════════════════════════
batch = 8

# ── Regression losses ────────────────────────────────────────────────────────
pred_reg = torch.randn(batch, 1)
true_reg = torch.randn(batch, 1)
print("MSE  :", nn.MSELoss()(pred_reg, true_reg).item())
print("MAE  :", nn.L1Loss()(pred_reg, true_reg).item())
print("Huber:", nn.HuberLoss()(pred_reg, true_reg).item())
print("SmoothL1:", nn.SmoothL1Loss()(pred_reg, true_reg).item())

# ── Binary classification ─────────────────────────────────────────────────────
logits = torch.randn(batch)
labels_b = torch.randint(0, 2, (batch,)).float()
print("\nBCEWithLogits:", nn.BCEWithLogitsLoss()(logits, labels_b).item())
probs = torch.sigmoid(logits)
print("BCE          :", nn.BCELoss()(probs, labels_b).item())

# ── Multi-class classification ────────────────────────────────────────────────
logits_mc = torch.randn(batch, 10)
labels_mc = torch.randint(0, 10, (batch,))
print("\nCrossEntropy :", nn.CrossEntropyLoss()(logits_mc, labels_mc).item())
print("NLLLoss      :", nn.NLLLoss()(F.log_softmax(logits_mc, dim=1), labels_mc).item())

# ── Weighted CrossEntropy (handle class imbalance) ────────────────────────────
weights = torch.ones(10); weights[0] = 3.0  # class 0 weighted 3x
print("Weighted CE  :", nn.CrossEntropyLoss(weight=weights)(logits_mc, labels_mc).item())

# ── KL Divergence ────────────────────────────────────────────────────────────
q = F.log_softmax(torch.randn(batch, 10), dim=1)
p = F.softmax(torch.randn(batch, 10), dim=1)
print("\nKLDiv        :", nn.KLDivLoss(reduction='batchmean')(q, p).item())

# ══════════════════════════════════════════════════════════════════════════════
#  TENSORFLOW LOSS FUNCTIONS
# ══════════════════════════════════════════════════════════════════════════════
y_true = tf.constant([1,0,1,1,0], dtype=tf.float32)
y_pred = tf.constant([0.9,0.1,0.8,0.3,0.2])
print("\n--- TensorFlow ---")
print("MSE        :", keras.losses.MeanSquaredError()(y_true, y_pred).numpy())
print("MAE        :", keras.losses.MeanAbsoluteError()(y_true, y_pred).numpy())
print("BCE        :", keras.losses.BinaryCrossentropy()(y_true, y_pred).numpy())

y_true_mc = tf.constant([0,1,2,1,0])
y_pred_mc = tf.random.uniform((5,3))
y_pred_mc = tf.nn.softmax(y_pred_mc)
print("SparseCCE  :", keras.losses.SparseCategoricalCrossentropy()(y_true_mc, y_pred_mc).numpy())

---
## 7. 🚀 Optimizers

Optimizers update model parameters to minimize the loss.

| Optimizer | Key Idea | Best For |
|-----------|---------|----------|
| SGD | Plain gradient descent | Simple, with momentum |
| SGD+Momentum | Accelerated with velocity | CNNs, many SOTA results |
| RMSprop | Adaptive per-parameter LR | RNNs |
| Adam | Momentum + adaptive LR | General default |
| AdamW | Adam + weight decay fix | Transformers |
| Adagrad | Accumulates squared grads | Sparse data |
| LARS | Layer-wise LR | Large batch training |
| Lion | Simplified adaptive | Recent, memory efficient |

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PYTORCH OPTIMIZERS
# ══════════════════════════════════════════════════════════════════════════════
model = MLP(784, [256,128], 10).to(device)

# ── All common optimizers ─────────────────────────────────────────────────────
opt_sgd   = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)
opt_adam  = optim.Adam(model.parameters(), lr=1e-3, betas=(0.9, 0.999), eps=1e-8)
opt_adamw = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
opt_rms   = optim.RMSprop(model.parameters(), lr=1e-3, alpha=0.99)
opt_ada   = optim.Adagrad(model.parameters(), lr=0.01)
opt_nadam = optim.NAdam(model.parameters(), lr=1e-3)
opt_lbfgs = optim.LBFGS(model.parameters(), lr=1.0)  # for small models

# ── Per-layer learning rates (freeze/fine-tune) ───────────────────────────────
opt_perlayer = optim.Adam([
    {'params': model.net[0].parameters(), 'lr': 1e-4},  # lower LR for early layers
    {'params': model.net[-1].parameters(), 'lr': 1e-3}, # higher LR for last layer
], lr=5e-4)  # default LR for other params

# ── Standard optimization step ────────────────────────────────────────────────
def training_step(model, optimizer, x, y, criterion):
    optimizer.zero_grad()          # 1. Clear gradients
    output = model(x)             # 2. Forward pass
    loss = criterion(output, y)   # 3. Compute loss
    loss.backward()               # 4. Backward pass
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # gradient clip
    optimizer.step()              # 5. Update weights
    return loss.item()

print("Optimizers created successfully.")

# ══════════════════════════════════════════════════════════════════════════════
#  TENSORFLOW / KERAS OPTIMIZERS
# ══════════════════════════════════════════════════════════════════════════════
tf_sgd   = keras.optimizers.SGD(learning_rate=0.01, momentum=0.9, nesterov=True)
tf_adam  = keras.optimizers.Adam(learning_rate=1e-3, beta_1=0.9, beta_2=0.999)
tf_adamw = keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=0.01)
tf_rms   = keras.optimizers.RMSprop(learning_rate=1e-3, rho=0.9)
tf_ada   = keras.optimizers.Adagrad(learning_rate=0.01)
tf_nadam = keras.optimizers.Nadam(learning_rate=1e-3)
print("TF Optimizers OK")

---
## 8. 🔁 Complete Training Loop

The full pipeline: **Data → Forward → Loss → Backward → Update → Validate**

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PYTORCH — COMPLETE TRAINING LOOP (Production Grade)
# ══════════════════════════════════════════════════════════════════════════════

def train_pytorch(model, train_loader, val_loader, epochs=5, lr=1e-3):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(epochs):
        # ── TRAIN ──────────────────────────────────────────────────────────
        model.train()   # activates dropout, batchnorm training mode
        train_loss, correct, total = 0.0, 0, 0
        for batch_idx, (X, y) in enumerate(train_loader):
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)  # slightly faster than zero_()
            logits = model(X)
            loss = criterion(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item() * X.size(0)
            correct += (logits.argmax(1) == y).sum().item()
            total += X.size(0)

        train_loss /= total
        train_acc   = correct / total

        # ── VALIDATE ───────────────────────────────────────────────────────
        model.eval()    # disables dropout, uses running stats for batchnorm
        val_loss, vcorrect, vtotal = 0.0, 0, 0
        with torch.no_grad():   # no gradient computation
            for X, y in val_loader:
                X, y = X.to(device), y.to(device)
                logits = model(X)
                val_loss += criterion(logits, y).item() * X.size(0)
                vcorrect += (logits.argmax(1) == y).sum().item()
                vtotal += X.size(0)

        val_loss /= vtotal
        val_acc   = vcorrect / vtotal
        scheduler.step()

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        print(f"Epoch {epoch+1:02d}/{epochs} | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | "
              f"LR: {scheduler.get_last_lr()[0]:.6f}")

    return history

# ── Quick demo with synthetic data ────────────────────────────────────────────
X_fake = torch.randn(1000, 784)
y_fake = torch.randint(0, 10, (1000,))
ds_fake = TensorDataset(X_fake, y_fake)
train_ds, val_ds = random_split(ds_fake, [800, 200])
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
val_dl   = DataLoader(val_ds,   batch_size=64)

pt_model = MLP(784, [256, 128], 10)
hist = train_pytorch(pt_model, train_dl, val_dl, epochs=3)

# ── Plot training curves ──────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12,4))
ax1.plot(hist['train_loss'], label='Train'); ax1.plot(hist['val_loss'], label='Val')
ax1.set_title('Loss'); ax1.legend()
ax2.plot(hist['train_acc'], label='Train'); ax2.plot(hist['val_acc'], label='Val')
ax2.set_title('Accuracy'); ax2.legend()
plt.tight_layout(); plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  TENSORFLOW — COMPLETE TRAINING LOOP
# ══════════════════════════════════════════════════════════════════════════════

# ── Method 1: model.compile + model.fit (high-level) ─────────────────────────
tf_model = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

tf_model.compile(
    optimizer = keras.optimizers.AdamW(learning_rate=1e-3),
    loss      = 'sparse_categorical_crossentropy',
    metrics   = ['accuracy']
)

X_np = X_fake.numpy(); y_np = y_fake.numpy()
callbacks_list = [
    EarlyStopping(patience=3, restore_best_weights=True),
    ReduceLROnPlateau(factor=0.5, patience=2),
]
hist_tf = tf_model.fit(
    X_np, y_np,
    epochs=3, batch_size=64,
    validation_split=0.2,
    callbacks=callbacks_list,
    verbose=1
)

# ── Method 2: Custom training loop (like PyTorch) ─────────────────────────────
tf_model2 = MLPKeras([256, 128], 10)
optimizer_tf = keras.optimizers.Adam(1e-3)
loss_fn_tf   = keras.losses.SparseCategoricalCrossentropy()
train_acc_m  = keras.metrics.SparseCategoricalAccuracy()

@tf.function  # compile for speed
def train_step_tf(x_batch, y_batch):
    with tf.GradientTape() as tape:
        predictions = tf_model2(x_batch, training=True)
        loss = loss_fn_tf(y_batch, predictions)
    gradients = tape.gradient(loss, tf_model2.trainable_variables)
    optimizer_tf.apply_gradients(zip(gradients, tf_model2.trainable_variables))
    train_acc_m.update_state(y_batch, predictions)
    return loss

dataset_tf = tf.data.Dataset.from_tensor_slices((X_np, y_np)).shuffle(1000).batch(64)
for epoch in range(3):
    train_acc_m.reset_state()
    epoch_loss = 0.0
    for step, (xb, yb) in enumerate(dataset_tf):
        loss_val = train_step_tf(xb, yb)
        epoch_loss += loss_val.numpy()
    print(f"Epoch {epoch+1} | Loss: {epoch_loss/(step+1):.4f} | Acc: {train_acc_m.result().numpy():.4f}")

---
## 9. 📦 Datasets & DataLoaders

Efficient data pipelines are critical for training performance.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PYTORCH — CUSTOM DATASET + DATALOADER
# ══════════════════════════════════════════════════════════════════════════════

class CustomDataset(Dataset):
    """Template for any custom dataset."""
    def __init__(self, X, y, transform=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.transform = transform

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        sample = self.X[idx]
        if self.transform:
            sample = self.transform(sample)
        return sample, self.y[idx]

# ── DataLoader configuration ──────────────────────────────────────────────────
X_data = np.random.randn(1000, 784).astype(np.float32)
y_data = np.random.randint(0, 10, 1000)

dataset = CustomDataset(X_data, y_data)
train_size = int(0.8 * len(dataset))
val_size   = len(dataset) - train_size
train_set, val_set = random_split(dataset, [train_size, val_size],
                                  generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(
    train_set,
    batch_size   = 64,
    shuffle      = True,
    num_workers  = 0,          # >0 for parallel loading (use 4 on real tasks)
    pin_memory   = True,       # faster CPU->GPU transfer
    drop_last    = False,      # drop incomplete final batch if True
)
val_loader = DataLoader(val_set, batch_size=64, shuffle=False)

# ── Image transforms with torchvision ────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomResizedCrop(32, scale=(0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std =[0.229,0.224,0.225])  # ImageNet stats
])
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

# ── Built-in datasets ─────────────────────────────────────────────────────────
# MNIST, CIFAR10/100, ImageNet, COCO, etc.
mnist_train = torchvision.datasets.MNIST(root='./data', train=True,
                                          download=True,
                                          transform=transforms.ToTensor())
mnist_loader = DataLoader(mnist_train, batch_size=128, shuffle=True)
print(f"MNIST batches: {len(mnist_loader)} | Batch shape: {next(iter(mnist_loader))[0].shape}")

# ══════════════════════════════════════════════════════════════════════════════
#  TENSORFLOW — tf.data.Dataset API
# ══════════════════════════════════════════════════════════════════════════════

# Create dataset
tf_dataset = tf.data.Dataset.from_tensor_slices((X_data, y_data))

# Full pipeline
AUTOTUNE = tf.data.AUTOTUNE
tf_train_ds = (tf_dataset
    .shuffle(buffer_size=1000, seed=42)
    .batch(64)
    .map(lambda x, y: (x, y), num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE)   # overlap CPU prep with GPU compute
)

# From numpy / generators
def data_gen():
    for i in range(100):
        yield np.random.randn(784).astype(np.float32), np.random.randint(10)

tf_gen_ds = tf.data.Dataset.from_generator(
    data_gen,
    output_signature=(tf.TensorSpec(shape=(784,), dtype=tf.float32),
                      tf.TensorSpec(shape=(), dtype=tf.int32))
)

# Built-in datasets (tensorflow_datasets)
# import tensorflow_datasets as tfds
# ds = tfds.load('mnist', split='train', as_supervised=True)

print("TF dataset batches:", sum(1 for _ in tf_train_ds))

---
## 10. 🖼️ Convolutional Neural Networks (CNNs)

CNNs exploit **spatial locality** and **translation invariance** via convolutions.

```
Conv2d output size = ⌊(W - K + 2P) / S⌋ + 1
  W = input width, K = kernel, P = padding, S = stride
```

| Component | Purpose |
|-----------|---------|
| Conv2d | Feature extraction |
| MaxPool / AvgPool | Spatial downsampling |
| GlobalAvgPool | Collapse spatial dims |
| BatchNorm2d | Normalize feature maps |
| Skip connections | Residual learning |

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PYTORCH CNNs
# ══════════════════════════════════════════════════════════════════════════════

# ── Residual Block (the core of ResNet) ───────────────────────────────────────
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_channels)
        self.relu  = nn.ReLU(inplace=True)
        # shortcut to match dimensions
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)   # skip connection
        return self.relu(out)

# ── Full ResNet-like CNN ───────────────────────────────────────────────────────
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, 1, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True)
        )
        self.layer1 = ResidualBlock(32, 64, stride=2)
        self.layer2 = ResidualBlock(64, 128, stride=2)
        self.layer3 = ResidualBlock(128, 256, stride=2)
        self.gap    = nn.AdaptiveAvgPool2d(1)  # Global Average Pooling
        self.head   = nn.Linear(256, num_classes)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.gap(x).flatten(1)
        return self.head(x)

cnn = SimpleCNN(10).to(device)
x_img = torch.randn(4, 3, 32, 32).to(device)  # batch=4, RGB, 32x32
print("CNN output shape:", cnn(x_img).shape)    # (4, 10)

# ── Conv2d parameter reference ────────────────────────────────────────────────
# nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0,
#           dilation=1, groups=1, bias=True, padding_mode='zeros')

# ── Depthwise separable conv (MobileNet style) ────────────────────────────────
class DepthwiseSepConv(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.dw = nn.Conv2d(in_ch, in_ch, 3, stride, 1, groups=in_ch, bias=False)  # depthwise
        self.pw = nn.Conv2d(in_ch, out_ch, 1, bias=False)                           # pointwise
        self.bn = nn.BatchNorm2d(out_ch)
    def forward(self, x):
        return F.relu(self.bn(self.pw(self.dw(x))))

print("Depthwise Sep conv:", DepthwiseSepConv(32,64)(torch.randn(1,32,16,16)).shape)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  TENSORFLOW CNNs
# ══════════════════════════════════════════════════════════════════════════════

def res_block_tf(x, filters, stride=1):
    """Functional API residual block."""
    shortcut = x
    x = layers.Conv2D(filters, 3, strides=stride, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride, use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
    x = layers.Add()([x, shortcut])
    return layers.Activation('relu')(x)

def build_cnn_tf(num_classes=10, input_shape=(32,32,3)):
    inputs = keras.Input(shape=input_shape)
    x = layers.Conv2D(32, 3, padding='same', use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = res_block_tf(x, 64, stride=2)
    x = res_block_tf(x, 128, stride=2)
    x = res_block_tf(x, 256, stride=2)
    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return keras.Model(inputs, outputs, name='SimpleCNN')

cnn_tf = build_cnn_tf()
x_img_tf = tf.random.normal((4, 32, 32, 3))   # TF uses channels-last (NHWC)
print("TF CNN output:", cnn_tf(x_img_tf).shape)

# ── Key TF Conv layers ────────────────────────────────────────────────────────
# layers.Conv2D(filters, kernel_size, strides=(1,1), padding='valid'/'same')
# layers.DepthwiseConv2D(kernel_size, ...)
# layers.SeparableConv2D(filters, kernel_size, ...)  # depthwise + pointwise
# layers.MaxPooling2D(pool_size=(2,2), strides=None)
# layers.GlobalAveragePooling2D()
# layers.GlobalMaxPooling2D()

---
## 11. 🔄 Recurrent Neural Networks (RNN, LSTM, GRU, BiLSTM)

For **sequential data** (text, time series, audio).

```
LSTM Gates:
  Forget: f_t = σ(W_f·[h_{t-1}, x_t] + b_f)
  Input:  i_t = σ(W_i·[h_{t-1}, x_t] + b_i)
  Cell:   c_t = f_t⊙c_{t-1} + i_t⊙tanh(W_c·[h_{t-1},x_t])
  Output: o_t = σ(W_o·[h_{t-1}, x_t] + b_o)
          h_t = o_t ⊙ tanh(c_t)
```

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PYTORCH — RNN / LSTM / GRU
# ══════════════════════════════════════════════════════════════════════════════

# Input: (seq_len, batch, input_size)  or  (batch, seq_len, input_size) if batch_first=True
SEQ_LEN, BATCH, INPUT_SIZE, HIDDEN = 20, 8, 16, 64
x_seq = torch.randn(BATCH, SEQ_LEN, INPUT_SIZE)  # batch_first

# ── Vanilla RNN ───────────────────────────────────────────────────────────────
rnn = nn.RNN(INPUT_SIZE, HIDDEN, num_layers=2, batch_first=True,
             dropout=0.2, nonlinearity='tanh')
out_rnn, h_n = rnn(x_seq)
print("RNN out:", out_rnn.shape, "| h_n:", h_n.shape)

# ── LSTM ──────────────────────────────────────────────────────────────────────
lstm = nn.LSTM(INPUT_SIZE, HIDDEN, num_layers=2, batch_first=True,
               dropout=0.2, bidirectional=False)
out_lstm, (h_n, c_n) = lstm(x_seq)
print("LSTM out:", out_lstm.shape, "| h_n:", h_n.shape, "| c_n:", c_n.shape)

# ── Bidirectional LSTM ────────────────────────────────────────────────────────
bilstm = nn.LSTM(INPUT_SIZE, HIDDEN, num_layers=2, batch_first=True,
                 dropout=0.2, bidirectional=True)
out_bi, _ = bilstm(x_seq)
print("BiLSTM out:", out_bi.shape)  # (BATCH, SEQ_LEN, 2*HIDDEN)

# ── GRU ───────────────────────────────────────────────────────────────────────
gru = nn.GRU(INPUT_SIZE, HIDDEN, batch_first=True, bidirectional=True)
out_gru, h_gru = gru(x_seq)
print("GRU out:", out_gru.shape)

# ── Sequence classifier with LSTM ─────────────────────────────────────────────
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden, n_layers, n_classes):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm  = nn.LSTM(embed_dim, hidden, n_layers, batch_first=True,
                             dropout=0.3, bidirectional=True)
        self.fc    = nn.Linear(hidden * 2, n_classes)
        self.drop  = nn.Dropout(0.3)

    def forward(self, token_ids):  # (batch, seq)
        x = self.drop(self.embed(token_ids))      # (B, S, E)
        out, (h, _) = self.lstm(x)
        # Concatenate final forward + backward hidden states
        h = torch.cat([h[-2], h[-1]], dim=1)      # (B, 2H)
        return self.fc(self.drop(h))              # (B, C)

model_lstm = LSTMClassifier(10000, 128, 256, 2, 5)
tokens = torch.randint(0, 10000, (8, 50))  # batch=8, seq=50
print("\nLSTM Classifier out:", model_lstm(tokens).shape)  # (8, 5)

---
## 12. 🤖 Transformers & Attention

The dominant architecture for NLP, vision, and multimodal tasks.

```
Attention(Q,K,V) = softmax(QKᵀ / √d_k) · V

Multi-Head: concat heads → project
Transformer Block: LayerNorm → MHA → Add → LayerNorm → FFN → Add
```

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PYTORCH — TRANSFORMER FROM SCRATCH
# ══════════════════════════════════════════════════════════════════════════════

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_k    = d_model // n_heads
        self.n_heads = n_heads
        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)
        self.Wo = nn.Linear(d_model, d_model)
        self.drop = nn.Dropout(dropout)
        self.scale = self.d_k ** -0.5

    def forward(self, x, mask=None):  # x: (B, S, D)
        B, S, D = x.shape
        def split_heads(t):
            return t.view(B, S, self.n_heads, self.d_k).transpose(1,2)  # (B,H,S,d_k)
        Q, K, V = map(split_heads, [self.Wq(x), self.Wk(x), self.Wv(x)])
        scores = (Q @ K.transpose(-2,-1)) * self.scale    # (B,H,S,S)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = self.drop(F.softmax(scores, dim=-1))        # (B,H,S,S)
        out  = (attn @ V).transpose(1,2).contiguous().view(B, S, D)  # (B,S,D)
        return self.Wo(out), attn

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, ffn_dim, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadSelfAttention(d_model, n_heads, dropout)
        self.ffn  = nn.Sequential(
            nn.Linear(d_model, ffn_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model)
        )
        self.ln1  = nn.LayerNorm(d_model)
        self.ln2  = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attn_out, attn_weights = self.attn(self.ln1(x), mask)   # Pre-LN
        x = x + self.drop(attn_out)                              # residual
        x = x + self.drop(self.ffn(self.ln2(x)))                # FFN + residual
        return x, attn_weights

class GPTLike(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, max_len, ffn_dim, dropout=0.1):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb   = nn.Embedding(max_len, d_model)
        self.blocks    = nn.ModuleList([TransformerBlock(d_model, n_heads, ffn_dim, dropout)
                                        for _ in range(n_layers)])
        self.ln_final  = nn.LayerNorm(d_model)
        self.head      = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, tokens, mask=None):  # tokens: (B, S)
        B, S = tokens.shape
        pos  = torch.arange(S, device=tokens.device).unsqueeze(0)  # (1, S)
        x    = self.token_emb(tokens) + self.pos_emb(pos)          # (B, S, D)
        for block in self.blocks:
            x, _ = block(x, mask)
        return self.head(self.ln_final(x))  # (B, S, vocab)

gpt = GPTLike(vocab_size=10000, d_model=256, n_heads=8, n_layers=4,
               max_len=512, ffn_dim=1024)
tokens_in = torch.randint(0, 10000, (4, 64))  # batch=4, seq=64
logits = gpt(tokens_in)
print("Transformer output:", logits.shape)   # (4, 64, 10000)

# ── PyTorch built-in Transformer ──────────────────────────────────────────────
encoder_layer = nn.TransformerEncoderLayer(d_model=256, nhead=8, dim_feedforward=1024,
                                            dropout=0.1, batch_first=True)
transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=4)
src = torch.randn(4, 64, 256)
enc_out = transformer_encoder(src)
print("Built-in Encoder out:", enc_out.shape)  # (4, 64, 256)

---
## 13. 🎯 Transfer Learning & Fine-tuning

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PYTORCH — TRANSFER LEARNING STRATEGIES
# ══════════════════════════════════════════════════════════════════════════════

# ── Strategy 1: Feature Extractor (freeze all, train head) ────────────────────
backbone = models.resnet50(weights='IMAGENET1K_V2')
# Freeze all layers
for param in backbone.parameters():
    param.requires_grad = False
# Replace final classification head
in_features = backbone.fc.in_features  # 2048
backbone.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(in_features, 256),
    nn.ReLU(),
    nn.Linear(256, 10)   # our number of classes
)
# Only head params require grad
print("Trainable params (feature extractor):",
      sum(p.numel() for p in backbone.parameters() if p.requires_grad))

# ── Strategy 2: Fine-tuning (unfreeze later layers) ────────────────────────────
backbone2 = models.resnet50(weights='IMAGENET1K_V2')
# Freeze all
for param in backbone2.parameters():
    param.requires_grad = False
# Unfreeze layer4 and head
for param in backbone2.layer4.parameters():
    param.requires_grad = True
backbone2.fc = nn.Linear(2048, 10)
# Use different LRs: lower for pretrained, higher for new head
optimizer_ft = optim.AdamW([
    {'params': backbone2.layer4.parameters(), 'lr': 1e-5},
    {'params': backbone2.fc.parameters(),     'lr': 1e-3},
])
print("Trainable params (fine-tune):",
      sum(p.numel() for p in backbone2.parameters() if p.requires_grad))

# ── Available pretrained models ────────────────────────────────────────────────
# models.resnet{18,34,50,101,152}
# models.efficientnet_{b0..b7}
# models.vit_b_16, vit_l_16       (Vision Transformer)
# models.convnext_{tiny,small,base,large}
# models.mobilenet_v3_{small,large}
# models.densenet{121,169,201}
# models.swin_t, swin_b           (Swin Transformer)

# ══════════════════════════════════════════════════════════════════════════════
#  TENSORFLOW — TRANSFER LEARNING
# ══════════════════════════════════════════════════════════════════════════════
base_model_tf = keras.applications.EfficientNetV2B0(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)
base_model_tf.trainable = False  # Freeze

inputs_tf = keras.Input(shape=(224, 224, 3))
x_tf = base_model_tf(inputs_tf, training=False)
x_tf = layers.GlobalAveragePooling2D()(x_tf)
x_tf = layers.Dropout(0.3)(x_tf)
outputs_tf = layers.Dense(10, activation='softmax')(x_tf)
transfer_model = keras.Model(inputs_tf, outputs_tf)

# Fine-tune: unfreeze top layers
base_model_tf.trainable = True
for layer in base_model_tf.layers[:-20]:  # freeze all but last 20
    layer.trainable = False

transfer_model.compile(
    optimizer=keras.optimizers.Adam(1e-5),   # low LR for fine-tuning
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
print("TF Transfer model ready. Trainable layers:",
      sum(1 for l in transfer_model.layers if l.trainable))

# ── Available TF pretrained models ────────────────────────────────────────────
# keras.applications.{VGG16, VGG19, ResNet50, ResNet101, EfficientNetV2B{0-3},
#                      MobileNetV3Small, InceptionV3, Xception, DenseNet{121,169,201}}

---
## 14. 🛡️ Regularization Techniques

Prevent **overfitting** and improve generalization.

| Technique | Effect |
|-----------|--------|
| L1 (Lasso) | Sparse weights |
| L2 (Ridge/Weight Decay) | Small weights |
| Dropout | Random neuron deactivation |
| DropBlock | Structured dropout for CNNs |
| Data Augmentation | Increases effective dataset size |
| Label Smoothing | Soft targets, less overconfident |
| Mixup | Interpolate training samples |
| CutMix | Paste patches from another image |
| Early Stopping | Stop when val loss increases |

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  REGULARIZATION TECHNIQUES
# ══════════════════════════════════════════════════════════════════════════════

# ── L1 / L2 Regularization (PyTorch) ─────────────────────────────────────────
def l1_l2_loss(model, lambda_l1=1e-5, lambda_l2=1e-4):
    l1 = sum(p.abs().sum() for p in model.parameters())
    l2 = sum(p.pow(2).sum() for p in model.parameters())
    return lambda_l1 * l1 + lambda_l2 * l2

# In training loop: loss = criterion(preds, targets) + l1_l2_loss(model)
# Note: AdamW implements L2 correctly via weight_decay parameter

# ── Dropout variants ──────────────────────────────────────────────────────────
x_in = torch.randn(4, 256)
print("Dropout:      ", nn.Dropout(p=0.5)(x_in).shape)
print("AlphaDropout: ", nn.AlphaDropout(p=0.1)(x_in).shape)  # for SELU
x_2d = torch.randn(4, 64, 8, 8)
print("Dropout2d:    ", nn.Dropout2d(p=0.2)(x_2d).shape)  # channel dropout

# ── Label Smoothing ───────────────────────────────────────────────────────────
criterion_ls = nn.CrossEntropyLoss(label_smoothing=0.1)
logits_demo = torch.randn(8, 10)
labels_demo = torch.randint(0, 10, (8,))
print("\nSmoothed CE:", criterion_ls(logits_demo, labels_demo).item())
print("Hard CE:    ", nn.CrossEntropyLoss()(logits_demo, labels_demo).item())

# ── Mixup augmentation ────────────────────────────────────────────────────────
def mixup(x, y, alpha=0.4):
    """Mix two training samples."""
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0))
    mixed_x = lam * x + (1-lam) * x[idx]
    # Loss: lam*criterion(pred,y) + (1-lam)*criterion(pred,y[idx])
    return mixed_x, y, y[idx], lam

# ── CutMix ────────────────────────────────────────────────────────────────────
def cutmix(x, y, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    B, C, H, W = x.shape
    idx = torch.randperm(B)
    # Random bounding box
    cut_w = int(W * np.sqrt(1-lam))
    cut_h = int(H * np.sqrt(1-lam))
    cx = np.random.randint(W); cy = np.random.randint(H)
    x1 = max(0, cx - cut_w//2); x2 = min(W, cx + cut_w//2)
    y1 = max(0, cy - cut_h//2); y2 = min(H, cy + cut_h//2)
    mixed = x.clone()
    mixed[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    lam_adj = 1 - (x2-x1)*(y2-y1)/(H*W)
    return mixed, y, y[idx], lam_adj

# ── TF Regularization ─────────────────────────────────────────────────────────
from tensorflow.keras import regularizers
l2_reg_layer = layers.Dense(128, activation='relu',
                              kernel_regularizer=regularizers.L2(1e-4),
                              bias_regularizer=regularizers.L2(1e-4))
l1l2_layer   = layers.Dense(64, kernel_regularizer=regularizers.L1L2(l1=1e-5, l2=1e-4))

# TF Dropout
mc_dropout = layers.Dropout(0.3)  # training=True at inference for MC Dropout
print("\nAll regularization techniques demo'd.")

---
## 15. 📊 Normalization Layers

| Norm | Normalizes Over | Best For |
|------|----------------|----------|
| BatchNorm | (N, H, W) per channel | CNNs, large batches |
| LayerNorm | All features per sample | Transformers, RNNs |
| InstanceNorm | (H, W) per sample per channel | Style transfer |
| GroupNorm | Groups of channels per sample | Small batch CNNs |
| RMSNorm | Per-token RMS | LLaMA, modern LLMs |

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  NORMALIZATION LAYERS
# ══════════════════════════════════════════════════════════════════════════════

# PyTorch
x_2d = torch.randn(16, 64, 32, 32)   # (N, C, H, W)
x_1d = torch.randn(16, 512)           # (N, D)
x_3d = torch.randn(4, 20, 256)        # (B, S, D)

print("BatchNorm2d:  ", nn.BatchNorm2d(64)(x_2d).shape)
print("BatchNorm1d:  ", nn.BatchNorm1d(512)(x_1d).shape)
print("LayerNorm:    ", nn.LayerNorm(256)(x_3d).shape)         # normalize last dim
print("InstanceNorm: ", nn.InstanceNorm2d(64)(x_2d).shape)
print("GroupNorm:    ", nn.GroupNorm(8, 64)(x_2d).shape)       # 8 groups of 8 ch

# ── RMSNorm (used in LLaMA) ───────────────────────────────────────────────────
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d))
        self.eps = eps
    def forward(self, x):
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).sqrt()
        return self.g * x / rms

print("RMSNorm:      ", RMSNorm(256)(x_3d).shape)

# TensorFlow
x_tf = tf.random.normal((4, 20, 256))
print("\nTF LayerNorm: ", layers.LayerNormalization()(x_tf).shape)
x_img_tf = tf.random.normal((4, 32, 32, 64))
print("TF BatchNorm: ", layers.BatchNormalization()(x_img_tf).shape)
print("TF GroupNorm: ", layers.GroupNormalization(groups=8)(x_img_tf).shape)

---
## 16. 📈 Learning Rate Schedulers

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PYTORCH LR SCHEDULERS
# ══════════════════════════════════════════════════════════════════════════════
model_s = nn.Linear(10, 2)
opt = optim.Adam(model_s.parameters(), lr=1e-3)
EPOCHS = 50

schedulers = {
    'StepLR'          : optim.lr_scheduler.StepLR(opt, step_size=10, gamma=0.5),
    'MultiStepLR'     : optim.lr_scheduler.MultiStepLR(opt, milestones=[10,25,40], gamma=0.5),
    'ExponentialLR'   : optim.lr_scheduler.ExponentialLR(opt, gamma=0.95),
    'CosineAnnealing' : optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS),
    'CosineWarmRestarts': optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=10, T_mult=2),
    'ReduceOnPlateau' : optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', patience=5, factor=0.5),
    'OneCycleLR'      : optim.lr_scheduler.OneCycleLR(opt, max_lr=1e-2,
                                                      total_steps=EPOCHS, pct_start=0.3),
    'LinearWarmup'    : optim.lr_scheduler.LinearLR(opt, start_factor=0.01, total_iters=10),
}

# ── Visualize CosineAnnealing ─────────────────────────────────────────────────
opt2 = optim.Adam(model_s.parameters(), lr=1e-3)
sched = optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=EPOCHS)
lrs = []
for _ in range(EPOCHS):
    lrs.append(opt2.param_groups[0]['lr'])
    sched.step()

plt.figure(figsize=(8,3))
plt.plot(lrs, marker='o', markersize=3)
plt.title('Cosine Annealing LR'); plt.xlabel('Epoch'); plt.ylabel('LR'); plt.grid(True)
plt.tight_layout(); plt.show()

# ── Warmup + Cosine (custom) ──────────────────────────────────────────────────
def cosine_warmup_schedule(optimizer, warmup_steps, total_steps):
    def lr_lambda(step):
        if step < warmup_steps:
            return step / warmup_steps
        progress = (step - warmup_steps) / (total_steps - warmup_steps)
        return 0.5 * (1 + np.cos(np.pi * progress))
    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# ── TF LR Schedulers ──────────────────────────────────────────────────────────
cosine_decay_tf = keras.optimizers.schedules.CosineDecay(1e-3, decay_steps=1000)
exp_decay_tf    = keras.optimizers.schedules.ExponentialDecay(1e-3, 1000, 0.96)
poly_decay_tf   = keras.optimizers.schedules.PolynomialDecay(1e-3, 1000, 1e-5)
warmup_cos_tf   = keras.optimizers.schedules.CosineDecayRestarts(1e-3, first_decay_steps=200)
print("LR at step 100:", cosine_decay_tf(100).numpy())

---
## 17. 💾 Model Saving & Loading

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PYTORCH — SAVE & LOAD
# ══════════════════════════════════════════════════════════════════════════════
model_save = MLP(784, [256], 10)

# ── Method 1: Save/Load state dict (RECOMMENDED) ─────────────────────────────
torch.save(model_save.state_dict(), 'model_weights.pth')
model_load = MLP(784, [256], 10)
model_load.load_state_dict(torch.load('model_weights.pth', map_location=device))
model_load.eval()
print("Model loaded from state_dict.")

# ── Method 2: Save entire model (not recommended — fragile) ──────────────────
torch.save(model_save, 'full_model.pth')
full_model = torch.load('full_model.pth')

# ── Method 3: Checkpoint (training state) ────────────────────────────────────
optimizer_ckpt = optim.Adam(model_save.parameters())
checkpoint = {
    'epoch'      : 10,
    'model_state': model_save.state_dict(),
    'optim_state': optimizer_ckpt.state_dict(),
    'train_loss' : 0.234,
    'val_acc'    : 0.91,
}
torch.save(checkpoint, 'checkpoint.pth')

# Resume training
ckpt = torch.load('checkpoint.pth')
model_save.load_state_dict(ckpt['model_state'])
optimizer_ckpt.load_state_dict(ckpt['optim_state'])
start_epoch = ckpt['epoch'] + 1
print(f"Resuming from epoch {start_epoch}")

# ── Partial loading (transfer learning) ──────────────────────────────────────
ckpt_state = torch.load('model_weights.pth')
model_new = MLP(784, [256], 10)
model_new.load_state_dict(ckpt_state, strict=False)  # strict=False ignores mismatches

# ══════════════════════════════════════════════════════════════════════════════
#  TENSORFLOW — SAVE & LOAD
# ══════════════════════════════════════════════════════════════════════════════
tf_model_sv = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(784,)),
    layers.Dense(10, activation='softmax')
])

# Method 1: SavedModel format (recommended)
tf_model_sv.save('my_model_tf')  # saves to directory
loaded_tf = keras.models.load_model('my_model_tf')

# Method 2: HDF5 format
tf_model_sv.save('my_model.h5')
loaded_h5 = keras.models.load_model('my_model.h5')

# Method 3: Weights only
tf_model_sv.save_weights('weights.h5')
tf_model_sv.load_weights('weights.h5')

# ModelCheckpoint callback (auto-saves best)
checkpoint_cb = ModelCheckpoint(
    'best_model.keras',
    monitor='val_accuracy', mode='max',
    save_best_only=True, verbose=1
)
print("Save/Load demos completed.")

---
## 18. 🖥️ GPU Acceleration

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  GPU ACCELERATION
# ══════════════════════════════════════════════════════════════════════════════

# ── PyTorch GPU ───────────────────────────────────────────────────────────────
print("CUDA available:", torch.cuda.is_available())
print("GPU count     :", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU name      :", torch.cuda.get_device_name(0))
    print("Memory alloc  :", torch.cuda.memory_allocated(0) / 1e9, 'GB')

# Move tensors/models to GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tensor_gpu = torch.randn(100, 100).to(device)      # via .to(device)
tensor_gpu2 = torch.randn(100, 100).cuda()         # direct .cuda()
model_gpu = MLP(784, [256], 10).to(device)
tensor_back = tensor_gpu.cpu()                      # move back to CPU

# ── Mixed Precision Training (FP16) ──────────────────────────────────────────
scaler = GradScaler()   # scales loss to avoid underflow in FP16

def train_step_amp(model, optimizer, x, y, criterion):
    optimizer.zero_grad()
    with autocast():                        # FP16 forward pass
        output = model(x)
        loss   = criterion(output, y)
    scaler.scale(loss).backward()           # scaled backward
    scaler.unscale_(optimizer)              # unscale before clip
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)                  # optimizer step
    scaler.update()                         # update scale factor
    return loss.item()

# ── Multi-GPU DataParallel ────────────────────────────────────────────────────
if torch.cuda.device_count() > 1:
    model_multi = nn.DataParallel(model_gpu)   # splits batch across GPUs

# ── DistributedDataParallel (preferred for multi-GPU) ────────────────────────
# torch.distributed.init_process_group(backend='nccl')
# model_ddp = nn.parallel.DistributedDataParallel(model, device_ids=[local_rank])
# Typically run via:  torchrun --nproc_per_node=4 train.py

# ── TF GPU config ─────────────────────────────────────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    # Memory growth — avoid allocating all GPU memory at once
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

# TF Mixed Precision
from tensorflow.keras import mixed_precision
# mixed_precision.set_global_policy('mixed_float16')  # uncomment to enable

# TF Multi-GPU
strategy = tf.distribute.MirroredStrategy()  # synchronous multi-GPU
print("TF replicas:", strategy.num_replicas_in_sync)
with strategy.scope():
    dist_model = keras.Sequential([
        layers.Dense(128, activation='relu', input_shape=(784,)),
        layers.Dense(10, activation='softmax')
    ])
print("GPU / Mixed Precision setup complete.")

---
## 19. 🎨 Generative Models: VAE & GAN

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  VARIATIONAL AUTOENCODER (VAE) — PyTorch
# ══════════════════════════════════════════════════════════════════════════════

class VAE(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=400, latent_dim=20):
        super().__init__()
        # Encoder
        self.enc_fc  = nn.Linear(input_dim, hidden_dim)
        self.enc_mu  = nn.Linear(hidden_dim, latent_dim)   # mean
        self.enc_log = nn.Linear(hidden_dim, latent_dim)   # log variance
        # Decoder
        self.dec_fc1 = nn.Linear(latent_dim, hidden_dim)
        self.dec_fc2 = nn.Linear(hidden_dim, input_dim)

    def encode(self, x):
        h = F.relu(self.enc_fc(x))
        return self.enc_mu(h), self.enc_log(h)

    def reparameterize(self, mu, logvar):
        """z = mu + eps * std  (reparameterization trick)"""
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = F.relu(self.dec_fc1(z))
        return torch.sigmoid(self.dec_fc2(h))

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

def vae_loss(recon_x, x, mu, logvar, beta=1.0):
    """ELBO = Reconstruction Loss + β*KL Divergence"""
    recon_loss = F.binary_cross_entropy(recon_x, x, reduction='sum')
    kl_div = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + beta * kl_div

vae = VAE(784, 400, 20)
x_test = torch.rand(8, 784)
recon, mu, logvar = vae(x_test)
loss_v = vae_loss(recon, x_test, mu, logvar)
print(f"VAE  | Input: {x_test.shape} | Recon: {recon.shape} | Loss: {loss_v.item():.2f}")

# ══════════════════════════════════════════════════════════════════════════════
#  GENERATIVE ADVERSARIAL NETWORK (GAN) — PyTorch
# ══════════════════════════════════════════════════════════════════════════════

class Generator(nn.Module):
    def __init__(self, latent_dim=100, out_dim=784):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 256), nn.LeakyReLU(0.2), nn.BatchNorm1d(256),
            nn.Linear(256, 512),        nn.LeakyReLU(0.2), nn.BatchNorm1d(512),
            nn.Linear(512, out_dim),    nn.Tanh()
        )
    def forward(self, z): return self.net(z)

class Discriminator(nn.Module):
    def __init__(self, in_dim=784):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 512), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(512, 256),    nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(256, 1)       # no sigmoid — use BCEWithLogitsLoss
        )
    def forward(self, x): return self.net(x)

G = Generator(100, 784)
D = Discriminator(784)
opt_G = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_D = optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
gan_loss = nn.BCEWithLogitsLoss()

def gan_train_step(real_imgs, batch_size=32, latent_dim=100):
    real_imgs = real_imgs.view(batch_size, -1)
    real_labels = torch.ones(batch_size, 1)
    fake_labels = torch.zeros(batch_size, 1)

    # Train Discriminator
    z = torch.randn(batch_size, latent_dim)
    fake_imgs = G(z).detach()  # don't backprop through G
    d_loss = (gan_loss(D(real_imgs), real_labels) +
              gan_loss(D(fake_imgs), fake_labels)) / 2
    opt_D.zero_grad(); d_loss.backward(); opt_D.step()

    # Train Generator
    z = torch.randn(batch_size, latent_dim)
    g_loss = gan_loss(D(G(z)), real_labels)  # fool D
    opt_G.zero_grad(); g_loss.backward(); opt_G.step()

    return d_loss.item(), g_loss.item()

# Demo step
fake_real = torch.randn(32, 784)
d_l, g_l = gan_train_step(fake_real)
print(f"GAN  | D Loss: {d_l:.4f} | G Loss: {g_l:.4f}")

---
## 20. 📝 Natural Language Processing (NLP)

Key NLP operations in deep learning: tokenization, embeddings, sequence modeling.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  NLP WITH PYTORCH
# ══════════════════════════════════════════════════════════════════════════════

# ── Embeddings ────────────────────────────────────────────────────────────────
VOCAB_SIZE, EMBED_DIM = 10000, 128
embedding = nn.Embedding(VOCAB_SIZE, EMBED_DIM, padding_idx=0)
# Load pretrained (e.g., GloVe)
# pretrained_vectors = torch.FloatTensor(glove_matrix)
# embedding = nn.Embedding.from_pretrained(pretrained_vectors, freeze=False)

tokens = torch.randint(1, VOCAB_SIZE, (8, 50))  # batch=8, seq_len=50
embedded = embedding(tokens)
print("Embedded shape:", embedded.shape)  # (8, 50, 128)

# ── Positional Encoding (Sinusoidal) ──────────────────────────────────────────
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.drop = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * -(np.log(10000.0)/d_model))
        pe[:, 0::2] = torch.sin(pos * div)   # even dims
        pe[:, 1::2] = torch.cos(pos * div)   # odd dims
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):  # x: (B, S, D)
        return self.drop(x + self.pe[:, :x.size(1)])

pos_enc = PositionalEncoding(128)
print("Positional encoded:", pos_enc(embedded).shape)

# ── Attention mask (for padding) ─────────────────────────────────────────────
def make_pad_mask(token_ids, pad_idx=0):
    """True where NOT padding (1=attend, 0=ignore)"""
    return (token_ids != pad_idx).unsqueeze(1).unsqueeze(2)  # (B,1,1,S)

# ── Causal mask (for autoregressive/decoder) ─────────────────────────────────
def make_causal_mask(seq_len):
    """Lower triangular mask — prevents attending to future tokens."""
    return torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0).unsqueeze(0)

# ── HuggingFace Transformers integration ──────────────────────────────────────
# from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
# tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
# model_hf  = AutoModel.from_pretrained('bert-base-uncased')
# inputs    = tokenizer(texts, padding=True, truncation=True, return_tensors='pt')
# outputs   = model_hf(**inputs)
# cls_embed = outputs.last_hidden_state[:, 0, :]  # [CLS] token embedding

# ── Text Classification head ──────────────────────────────────────────────────
class TextClassifier(nn.Module):
    def __init__(self, vocab, embed_dim, d_model, n_heads, n_layers, n_classes):
        super().__init__()
        self.embed = nn.Embedding(vocab, embed_dim, padding_idx=0)
        self.proj  = nn.Linear(embed_dim, d_model)
        self.pe    = PositionalEncoding(d_model)
        enc_layer  = nn.TransformerEncoderLayer(d_model, n_heads, dim_feedforward=4*d_model,
                                                 batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, n_layers)
        self.head    = nn.Linear(d_model, n_classes)

    def forward(self, tokens, key_padding_mask=None):
        x = self.pe(self.proj(self.embed(tokens)))
        # key_padding_mask: True where PADDING (opposite of our attend mask)
        x = self.encoder(x, src_key_padding_mask=key_padding_mask)
        return self.head(x.mean(dim=1))  # mean pooling over sequence

clf = TextClassifier(10000, 128, 256, 8, 4, 3)
out = clf(tokens)
print("Text Classifier output:", out.shape)  # (8, 3)

---
## 21. 📊 Evaluation Metrics

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  EVALUATION METRICS
# ══════════════════════════════════════════════════════════════════════════════
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report)

# Simulated predictions
y_true_np = np.random.randint(0, 3, 100)
y_pred_np = np.random.randint(0, 3, 100)
y_prob_np = np.random.dirichlet(np.ones(3), size=100)  # soft probabilities

print("=== Classification Metrics ===")
print(f"Accuracy   : {accuracy_score(y_true_np, y_pred_np):.4f}")
print(f"Precision  : {precision_score(y_true_np, y_pred_np, average='macro', zero_division=0):.4f}")
print(f"Recall     : {recall_score(y_true_np, y_pred_np, average='macro', zero_division=0):.4f}")
print(f"F1 Score   : {f1_score(y_true_np, y_pred_np, average='macro', zero_division=0):.4f}")
print(f"ROC-AUC    : {roc_auc_score(y_true_np, y_prob_np, multi_class='ovr'):.4f}")
print("\nClassification Report:\n", classification_report(y_true_np, y_pred_np, zero_division=0))

# ── Confusion Matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(y_true_np, y_pred_np)
fig, ax = plt.subplots(figsize=(5,4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix')
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i,j], ha='center', va='center', color='black')
plt.colorbar(im); plt.tight_layout(); plt.show()

# ── PyTorch inline metrics ────────────────────────────────────────────────────
def accuracy(logits, targets):
    return (logits.argmax(1) == targets).float().mean().item()

def top_k_accuracy(logits, targets, k=5):
    topk = logits.topk(k, dim=1).indices
    return (topk == targets.unsqueeze(1)).any(dim=1).float().mean().item()

# ── TF Metrics ────────────────────────────────────────────────────────────────
# Cumulative metrics (update in training loop)
acc_metric  = keras.metrics.SparseCategoricalAccuracy()
auc_metric  = keras.metrics.AUC(multi_label=False)
prec_metric = keras.metrics.Precision()
rec_metric  = keras.metrics.Recall()

# Usage in loop:
# acc_metric.update_state(y_true, y_pred)
# print(acc_metric.result().numpy())
# acc_metric.reset_state()
print("\nMetrics demo complete.")

---
## 22. ⚙️ Custom Layers & Weight Initialization

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CUSTOM LAYERS & WEIGHT INIT
# ══════════════════════════════════════════════════════════════════════════════

# ── Custom Attention Layer ────────────────────────────────────────────────────
class ScaledDotProductAttention(nn.Module):
    def __init__(self, d_k, dropout=0.0):
        super().__init__()
        self.scale = d_k ** -0.5
        self.drop  = nn.Dropout(dropout)

    def forward(self, Q, K, V, mask=None):
        scores = torch.matmul(Q, K.transpose(-2,-1)) * self.scale
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = self.drop(F.softmax(scores, dim=-1))
        return torch.matmul(attn, V), attn

# ── Custom Swish Activation ───────────────────────────────────────────────────
class Swish(nn.Module):
    def forward(self, x): return x * torch.sigmoid(x)

# ── Custom Conv + BN + Act block ─────────────────────────────────────────────
class ConvBnAct(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1, act=nn.ReLU(inplace=True)):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, k, s, p, bias=False),
            nn.BatchNorm2d(out_ch), act
        )
    def forward(self, x): return self.block(x)

# ── Custom loss function ──────────────────────────────────────────────────────
class FocalLoss(nn.Module):
    """Focal Loss for class imbalance (RetinaNet)."""
    def __init__(self, gamma=2.0, alpha=0.25):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, logits, targets):
        ce    = F.cross_entropy(logits, targets, reduction='none')
        pt    = torch.exp(-ce)
        focal = self.alpha * (1-pt)**self.gamma * ce
        return focal.mean()

# ── Weight Initialization ─────────────────────────────────────────────────────
def init_weights(model):
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)          # Xavier for sigmoid/tanh
            # nn.init.kaiming_normal_(m.weight, nonlinearity='relu')  # He for ReLU
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        elif isinstance(m, (nn.BatchNorm2d, nn.LayerNorm)):
            nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0, std=0.02)  # GPT-style

# Apply initialization
new_model = MLP(784, [256], 10)
init_weights(new_model)

# Test custom layers
focal = FocalLoss()
logits_test = torch.randn(8, 5)
targets_test = torch.randint(0, 5, (8,))
print("Focal Loss:", focal(logits_test, targets_test).item())

# ── TF Custom Layer ───────────────────────────────────────────────────────────
class CustomDenseLayer(keras.layers.Layer):
    def __init__(self, units, activation=None):
        super().__init__()
        self.units = units
        self.activation = keras.activations.get(activation)

    def build(self, input_shape):
        self.W = self.add_weight('W', (input_shape[-1], self.units),
                                  initializer='glorot_uniform', trainable=True)
        self.b = self.add_weight('b', (self.units,),
                                  initializer='zeros', trainable=True)

    def call(self, x):
        out = tf.matmul(x, self.W) + self.b
        return self.activation(out) if self.activation else out

custom_layer = CustomDenseLayer(64, activation='relu')
x_tf = tf.random.normal((8, 128))
print("Custom TF layer:", custom_layer(x_tf).shape)

---
## 23. 🔍 Debugging & Visualization

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  DEBUGGING & VISUALIZATION TOOLS
# ══════════════════════════════════════════════════════════════════════════════

# ── Gradient flow check ───────────────────────────────────────────────────────
def check_gradients(model):
    print(f"{'Layer':<40} {'Grad norm':>12}")
    print('-'*54)
    for name, param in model.named_parameters():
        if param.grad is not None:
            grad_norm = param.grad.norm().item()
            flag = " ⚠️ VANISHING" if grad_norm < 1e-6 else (" ⚠️ EXPLODING" if grad_norm > 100 else "")
            print(f"{name:<40} {grad_norm:>12.6f}{flag}")
        else:
            print(f"{name:<40} {'NO GRAD':>12}")

# ── Hook to capture activations ──────────────────────────────────────────────
class ActivationHook:
    def __init__(self):
        self.activations = {}

    def hook_fn(self, name):
        def hook(module, input, output):
            self.activations[name] = output.detach()
        return hook

    def register(self, model):
        for name, layer in model.named_modules():
            if isinstance(layer, (nn.ReLU, nn.Linear)):
                layer.register_forward_hook(self.hook_fn(name))

hook = ActivationHook()
demo_model = MLP(784, [256, 128], 10)
hook.register(demo_model)
_ = demo_model(torch.randn(4, 784))
print("Captured activations:", list(hook.activations.keys()))

# ── Plot weight distributions ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, param) in zip(axes, list(demo_model.named_parameters())[:3]):
    ax.hist(param.data.numpy().flatten(), bins=50, color='steelblue', alpha=0.7)
    ax.set_title(f'{name}\nshape={list(param.shape)}')
    ax.set_xlabel('Weight value')
plt.suptitle('Weight Distributions'); plt.tight_layout(); plt.show()

# ── Check for NaN / Inf in outputs ───────────────────────────────────────────
def check_tensor(t, name='tensor'):
    has_nan = torch.isnan(t).any().item()
    has_inf = torch.isinf(t).any().item()
    print(f"{name}: NaN={has_nan}, Inf={has_inf}, min={t.min():.4f}, max={t.max():.4f}")

check_tensor(torch.randn(100), 'normal')

# ── TensorBoard integration ───────────────────────────────────────────────────
from torch.utils.tensorboard import SummaryWriter
# writer = SummaryWriter('runs/experiment_1')
# for epoch, (loss, acc) in enumerate(zip(losses, accs)):
#     writer.add_scalar('Loss/train', loss, epoch)
#     writer.add_scalar('Accuracy/train', acc, epoch)
# writer.add_graph(model, input_to_model=torch.randn(1, 784))
# writer.add_histogram('layer/weights', model.net[0].weight, epoch)
# writer.close()
# # Launch: tensorboard --logdir=runs
print("\nDebugging tools ready. TensorBoard: tensorboard --logdir=runs")

---
## 24. ⚗️ Hyperparameter Tuning

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  HYPERPARAMETER SEARCH
# ══════════════════════════════════════════════════════════════════════════════

# ── Manual grid search ────────────────────────────────────────────────────────
from itertools import product

hp_grid = {
    'lr'       : [1e-4, 1e-3, 1e-2],
    'dropout'  : [0.1, 0.3, 0.5],
    'hidden'   : [128, 256, 512],
    'batch_size': [32, 64, 128]
}

best_val_acc, best_config = 0.0, {}
print(f"Total combinations: {np.prod([len(v) for v in hp_grid.values()])}")

# ── Optuna (automatic Bayesian optimization — RECOMMENDED) ────────────────────
# pip install optuna
# import optuna
#
# def objective(trial):
#     lr       = trial.suggest_float('lr', 1e-5, 1e-2, log=True)
#     dropout  = trial.suggest_float('dropout', 0.0, 0.5)
#     n_hidden = trial.suggest_int('n_hidden', 64, 512)
#     n_layers = trial.suggest_int('n_layers', 1, 4)
#     optimizer_name = trial.suggest_categorical('optimizer', ['Adam', 'AdamW', 'SGD'])
#
#     model = MLP(784, [n_hidden]*n_layers, 10).to(device)
#     optimizer = getattr(optim, optimizer_name)(model.parameters(), lr=lr)
#     # ... train and return val_accuracy
#     return val_accuracy
#
# study = optuna.create_study(direction='maximize')
# study.optimize(objective, n_trials=100, n_jobs=4)
# print('Best trial:', study.best_trial.params)

# ── Keras Tuner (for TF) ──────────────────────────────────────────────────────
# pip install keras-tuner
# import keras_tuner as kt
#
# def build_model(hp):
#     model = keras.Sequential()
#     model.add(layers.Input(shape=(784,)))
#     for i in range(hp.Int('n_layers', 1, 4)):
#         model.add(layers.Dense(
#             units=hp.Int(f'units_{i}', 64, 512, step=64),
#             activation=hp.Choice('activation', ['relu', 'gelu'])
#         ))
#         model.add(layers.Dropout(hp.Float('dropout', 0.0, 0.5)))
#     model.add(layers.Dense(10, activation='softmax'))
#     model.compile(
#         optimizer=keras.optimizers.Adam(hp.Float('lr', 1e-5, 1e-2, sampling='log')),
#         loss='sparse_categorical_crossentropy', metrics=['accuracy'])
#     return model
#
# tuner = kt.BayesianOptimization(build_model, objective='val_accuracy', max_trials=50)
# tuner.search(X_train, y_train, validation_data=(X_val, y_val), epochs=10)
# best_model = tuner.get_best_models(1)[0]

print("Hyperparameter tuning reference ready.")
print("Recommended: Optuna for PyTorch, KerasTuner for TF.")

---
## 25. 🚢 Model Deployment & ONNX

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  MODEL DEPLOYMENT
# ══════════════════════════════════════════════════════════════════════════════

# ── TorchScript (serialize PyTorch model) ────────────────────────────────────
model_deploy = MLP(784, [256], 10).eval()

# Method 1: Tracing (good for simple models)
example_input = torch.randn(1, 784)
traced_model = torch.jit.trace(model_deploy, example_input)
traced_model.save('traced_model.pt')

# Method 2: Scripting (good for models with control flow)
scripted_model = torch.jit.script(model_deploy)
scripted_model.save('scripted_model.pt')

# Load and run
loaded_ts = torch.jit.load('traced_model.pt')
with torch.no_grad():
    ts_out = loaded_ts(example_input)
print("TorchScript output:", ts_out.shape)

# ── Export to ONNX ────────────────────────────────────────────────────────────
torch.onnx.export(
    model_deploy,
    example_input,
    'model.onnx',
    export_params=True,
    opset_version=17,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)
print("ONNX model saved.")

# Run ONNX model
# import onnxruntime as ort
# ort_session = ort.InferenceSession('model.onnx')
# ort_inputs  = {ort_session.get_inputs()[0].name: example_input.numpy()}
# ort_out     = ort_session.run(None, ort_inputs)[0]

# ── TF Model Deployment ───────────────────────────────────────────────────────
# TFLite (mobile/edge)
# converter = tf.lite.TFLiteConverter.from_keras_model(tf_model)
# converter.optimizations = [tf.lite.Optimize.DEFAULT]  # quantization
# tflite_model = converter.convert()
# with open('model.tflite', 'wb') as f:
#     f.write(tflite_model)

# TF Serving (REST API)
# tf.saved_model.save(model, 'saved_model/1')
# docker run -p 8501:8501 --model_base_path=/models/model tensorflow/serving
# POST http://localhost:8501/v1/models/model:predict

# ── FastAPI serving example ───────────────────────────────────────────────────
SERVING_CODE = '''
# serve.py
from fastapi import FastAPI
import torch, numpy as np
from pydantic import BaseModel

app = FastAPI()
model = torch.jit.load('traced_model.pt').eval()

class PredictRequest(BaseModel):
    features: list[float]

@app.post('/predict')
async def predict(req: PredictRequest):
    x = torch.tensor(req.features).unsqueeze(0)
    with torch.no_grad():
        logits = model(x)
    probs = torch.softmax(logits, dim=1).squeeze().tolist()
    return {'class': int(logits.argmax(1)), 'probabilities': probs}

# Run: uvicorn serve:app --host 0.0.0.0 --port 8000
'''
print(SERVING_CODE)

---
## 26. 🎭 Diffusion Models (Score-Matching / DDPM Concepts)
State-of-the-art generative models for images, audio, and more.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  DDPM — DENOISING DIFFUSION PROBABILISTIC MODEL (Concepts + Code)
# ══════════════════════════════════════════════════════════════════════════════

class DiffusionScheduler:
    """
    Linear noise schedule: β_t linearly increases from β_start to β_end.
    Forward process: q(x_t | x_{t-1}) = N(x_t; √(1-β_t)·x_{t-1}, β_t·I)
    """
    def __init__(self, T=1000, beta_start=1e-4, beta_end=0.02):
        self.T = T
        self.beta   = torch.linspace(beta_start, beta_end, T)
        self.alpha  = 1.0 - self.beta
        self.alpha_bar = torch.cumprod(self.alpha, dim=0)  # ᾱ_t = ∏α_s

    def q_sample(self, x0, t):
        """Sample x_t given x_0: x_t = √ᾱ_t·x_0 + √(1-ᾱ_t)·ε"""
        a_bar = self.alpha_bar[t].view(-1,1)  # (B,1)
        noise = torch.randn_like(x0)
        return a_bar.sqrt() * x0 + (1-a_bar).sqrt() * noise, noise

    @torch.no_grad()
    def p_sample(self, model, x_t, t):
        """One denoising step: sample x_{t-1} from p_θ(x_{t-1}|x_t)"""
        beta_t   = self.beta[t].view(-1,1)
        a_t      = self.alpha[t].view(-1,1)
        a_bar_t  = self.alpha_bar[t].view(-1,1)
        t_tensor = torch.full((x_t.size(0),), t, dtype=torch.long)
        eps_pred = model(x_t, t_tensor)
        mean = (x_t - beta_t / (1-a_bar_t).sqrt() * eps_pred) / a_t.sqrt()
        if t > 0:
            z = torch.randn_like(x_t)
            return mean + beta_t.sqrt() * z
        return mean

class SimpleUNet(nn.Module):
    """Minimal U-Net for diffusion (real ones use attention, larger channels)"""
    def __init__(self, dim=64, t_emb_dim=128):
        super().__init__()
        # Time embedding
        self.t_emb = nn.Sequential(nn.Linear(1, t_emb_dim), nn.SiLU(),
                                    nn.Linear(t_emb_dim, t_emb_dim))
        # Encoder
        self.enc1 = nn.Linear(dim, 256)
        self.enc2 = nn.Linear(256 + t_emb_dim, 512)
        # Decoder
        self.dec1 = nn.Linear(512, 256)
        self.dec2 = nn.Linear(256 + 256, dim)  # skip connection
        self.act  = nn.SiLU()

    def forward(self, x, t):  # x: (B, dim), t: (B,)
        t_emb = self.t_emb(t.float().unsqueeze(-1) / 1000)  # normalize t
        h1 = self.act(self.enc1(x))
        h2 = self.act(self.enc2(torch.cat([h1, t_emb], dim=-1)))
        h3 = self.act(self.dec1(h2))
        return self.dec2(torch.cat([h3, h1], dim=-1))  # predict noise ε

# Training loop concept
scheduler = DiffusionScheduler(T=1000)
diff_model = SimpleUNet(64)
opt_diff   = optim.Adam(diff_model.parameters(), lr=1e-4)

x0 = torch.randn(8, 64)  # fake clean data
t  = torch.randint(0, 1000, (8,))  # random timestep per sample
xt, noise = scheduler.q_sample(x0, t)
noise_pred = diff_model(xt, t)
loss_diff  = F.mse_loss(noise_pred, noise)  # simple denoising objective
print(f"Diffusion | Noise pred shape: {noise_pred.shape} | Loss: {loss_diff.item():.4f}")
print("For production use: diffusers library (huggingface.co/docs/diffusers)")

---
## 27. 🏋️ Complete CIFAR-10 End-to-End Example

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  COMPLETE CIFAR-10 TRAINING PIPELINE
# ══════════════════════════════════════════════════════════════════════════════

# ── Data ──────────────────────────────────────────────────────────────────────
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

cifar_train_tf = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.AutoAugment(transforms.AutoAugmentPolicy.CIFAR10),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD)
])
cifar_val_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD)
])

cifar_train = torchvision.datasets.CIFAR10('./data', train=True,  download=True, transform=cifar_train_tf)
cifar_test  = torchvision.datasets.CIFAR10('./data', train=False, download=True, transform=cifar_val_tf)

cifar_train_loader = DataLoader(cifar_train, batch_size=128, shuffle=True,  num_workers=2, pin_memory=True)
cifar_test_loader  = DataLoader(cifar_test,  batch_size=256, shuffle=False, num_workers=2)

CLASSES = ('plane','car','bird','cat','deer','dog','frog','horse','ship','truck')

# ── Model: ResNet-like ─────────────────────────────────────────────────────────
def cifar_resnet():
    m = models.resnet18(weights=None)  # train from scratch
    m.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)  # smaller kernel for 32x32
    m.maxpool = nn.Identity()           # remove maxpool (CIFAR is small)
    m.fc = nn.Linear(512, 10)
    return m

model_cifar = cifar_resnet().to(device)
optimizer_cifar = optim.SGD(model_cifar.parameters(), lr=0.1,
                              momentum=0.9, weight_decay=5e-4)
scheduler_cifar = optim.lr_scheduler.OneCycleLR(
    optimizer_cifar, max_lr=0.1,
    steps_per_epoch=len(cifar_train_loader), epochs=5
)
criterion_cifar = nn.CrossEntropyLoss(label_smoothing=0.1)

# ── Training ───────────────────────────────────────────────────────────────────
def run_epoch(loader, train=True):
    model_cifar.train() if train else model_cifar.eval()
    total_loss = correct = total = 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            if train: optimizer_cifar.zero_grad(set_to_none=True)
            out  = model_cifar(X)
            loss = criterion_cifar(out, y)
            if train:
                loss.backward()
                optimizer_cifar.step()
                scheduler_cifar.step()
            total_loss += loss.item() * X.size(0)
            correct    += (out.argmax(1) == y).sum().item()
            total      += X.size(0)
    return total_loss/total, correct/total

print("Training CIFAR-10 (3 epochs demo)...")
for epoch in range(3):
    tr_loss, tr_acc = run_epoch(cifar_train_loader, train=True)
    val_loss, val_acc = run_epoch(cifar_test_loader, train=False)
    print(f"Epoch {epoch+1:2d} | Train: {tr_loss:.4f}/{tr_acc:.4f} | "
          f"Val: {val_loss:.4f}/{val_acc:.4f}")

---
## 28. 🧮 Distributed Training & Gradient Accumulation

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  GRADIENT ACCUMULATION (simulate larger batch)
# ══════════════════════════════════════════════════════════════════════════════

def train_with_grad_accum(model, loader, optimizer, criterion,
                           accumulation_steps=4, epochs=2):
    """Accumulate gradients over N mini-batches before updating."""
    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad(set_to_none=True)
        running_loss = 0.0
        for step, (X, y) in enumerate(loader):
            X, y = X.to(device), y.to(device)
            # Scale loss by accumulation steps to maintain consistent gradient magnitude
            loss = criterion(model(X), y) / accumulation_steps
            loss.backward()
            running_loss += loss.item()

            if (step+1) % accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)

        print(f"Epoch {epoch+1} loss: {running_loss / len(loader):.4f}")

# ── Distributed Training Template ─────────────────────────────────────────────
DDP_TEMPLATE = '''
# train_distributed.py  — run with: torchrun --nproc_per_node=4 train_distributed.py
import os, torch, torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler

def setup():
    dist.init_process_group('nccl')  # 'gloo' for CPU
    torch.cuda.set_device(int(os.environ['LOCAL_RANK']))

def cleanup():
    dist.destroy_process_group()

def main():
    setup()
    rank = dist.get_rank()
    device = torch.device(f'cuda:{rank}')

    model = YourModel().to(device)
    model = DDP(model, device_ids=[rank])  # wrap for DDP

    # DistributedSampler ensures each GPU gets different data
    sampler = DistributedSampler(dataset, num_replicas=dist.get_world_size(), rank=rank)
    loader  = DataLoader(dataset, sampler=sampler, batch_size=64)

    for epoch in range(EPOCHS):
        sampler.set_epoch(epoch)  # ensure different shuffle each epoch
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            loss = criterion(model(X), y)
            loss.backward()  # DDP auto-averages gradients across GPUs
            optimizer.step()

    # Save only from rank 0
    if rank == 0:
        torch.save(model.module.state_dict(), 'model.pth')
    cleanup()
'''
print(DDP_TEMPLATE)

# ── Run gradient accumulation demo ────────────────────────────────────────────
acc_model = MLP(784, [256], 10).to(device)
acc_opt   = optim.Adam(acc_model.parameters())
acc_crit  = nn.CrossEntropyLoss()
train_with_grad_accum(acc_model, train_dl, acc_opt, acc_crit, accumulation_steps=4)

---
## 29. 🌟 Advanced Architectures Reference

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  ARCHITECTURE QUICK REFERENCE
# ══════════════════════════════════════════════════════════════════════════════

# ── Vision Transformer (ViT) Patch Embedding ─────────────────────────────────
class PatchEmbedding(nn.Module):
    """Split image into patches and embed them (ViT core)."""
    def __init__(self, img_size=224, patch_size=16, in_channels=3, embed_dim=768):
        super().__init__()
        n_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim, patch_size, stride=patch_size)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches+1, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x):  # x: (B,3,H,W)
        B = x.size(0)
        x = self.proj(x).flatten(2).transpose(1,2)  # (B, N, D)
        cls = self.cls_token.expand(B,-1,-1)         # (B, 1, D)
        x = torch.cat([cls, x], dim=1)               # (B, N+1, D)
        return x + self.pos_embed

vit_patch = PatchEmbedding(224, 16, 3, 768)
img_in = torch.randn(2, 3, 224, 224)
print("ViT patch embed output:", vit_patch(img_in).shape)  # (2, 197, 768)

# ── Squeeze-Excitation Block (SENet) ─────────────────────────────────────────
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(channels, channels//reduction), nn.ReLU(),
            nn.Linear(channels//reduction, channels), nn.Sigmoid()
        )
    def forward(self, x):
        scale = self.se(x).view(x.size(0), x.size(1), 1, 1)
        return x * scale  # channel recalibration

se = SEBlock(64)
print("SE Block:", se(torch.randn(4,64,8,8)).shape)

# ── CBAM (Convolutional Block Attention Module) ────────────────────────────────
class ChannelAttention(nn.Module):
    def __init__(self, ch, r=16):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(ch, ch//r), nn.ReLU(), nn.Linear(ch//r, ch))
    def forward(self, x):
        avg = self.fc(x.mean([-2,-1]))
        mx  = self.fc(x.amax([-2,-1]))
        return x * torch.sigmoid(avg + mx).unsqueeze(-1).unsqueeze(-1)

# ── Summary table ─────────────────────────────────────────────────────────────
ARCH_TABLE = """
╔══════════════════╦══════════════════════════════╦═══════════════════════╗
║ Architecture     ║ Key Innovation               ║ Use Case              ║
╠══════════════════╬══════════════════════════════╬═══════════════════════╣
║ ResNet           ║ Skip connections             ║ Image classification  ║
║ EfficientNet     ║ Compound scaling             ║ SOTA image clf        ║
║ ViT              ║ Pure attention for vision    ║ Large-scale images    ║
║ Swin Transformer ║ Shifted window attention     ║ Dense prediction      ║
║ BERT             ║ Bidirectional masking        ║ NLU tasks             ║
║ GPT              ║ Autoregressive generation    ║ Text generation       ║
║ T5               ║ Text-to-Text framework       ║ NLP multi-task        ║
║ CLIP             ║ Contrastive vision-language  ║ Multimodal            ║
║ Stable Diffusion ║ Latent diffusion             ║ Image generation      ║
║ LLaMA/Mistral    ║ Efficient open LLMs          ║ Language modeling     ║
╚══════════════════╩══════════════════════════════╩═══════════════════════╝
"""
print(ARCH_TABLE)

---
## 30. 🏆 Expert Tips & Best Practices

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  EXPERT TIPS CHEAT SHEET
# ══════════════════════════════════════════════════════════════════════════════

TIPS = """
╔══════════════════════════════════════════════════════════════════════════════╗
║                     🏆 DEEP LEARNING EXPERT TIPS                           ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  DATA                                                                        ║
║  • Always normalize inputs (zero mean, unit std or [0,1])                   ║
║  • Augment aggressively; often bigger effect than architecture               ║
║  • Check class balance; use weighted sampler / focal loss if imbalanced      ║
║  • Validate your data pipeline before training (plot samples!)               ║
║                                                                              ║
║  TRAINING                                                                    ║
║  • Start with AdamW + cosine LR + warmup. Tune batch size, LR together.     ║
║  • Linear scaling rule: LR ∝ batch_size (scale LR when increasing batch)    ║
║  • Gradient clipping (norm=1.0) prevents exploding gradients                 ║
║  • mixed_precision (fp16/bf16) = 2x speedup + 50% memory                   ║
║  • Use torch.compile() (PyTorch 2.0+) for ~30% speedup                     ║
║  • Label smoothing (0.1) + dropout (0.1-0.5) fight overfitting              ║
║  • EMA (Exponential Moving Average) of weights → better generalization      ║
║                                                                              ║
║  DEBUGGING                                                                   ║
║  • Overfit 1 batch first — if loss doesn't go to ~0, there's a bug          ║
║  • Check gradient norms; dead < 1e-6, exploding > 100                       ║
║  • Always use model.eval() at inference (disables dropout/BN stats)          ║
║  • Use torch.autograd.detect_anomaly() to find NaN sources                  ║
║                                                                              ║
║  ARCHITECTURE                                                                ║
║  • Pre-LN (LayerNorm before attention) trains more stably than Post-LN      ║
║  • Skip connections are almost always beneficial                             ║
║  • Global Average Pooling > Flatten for CNNs (less parameters, more robust) ║
║  • Kaiming init for ReLU networks, Xavier/Glorot for tanh/sigmoid           ║
║                                                                              ║
║  TRANSFER LEARNING                                                           ║
║  • Always freeze BN when fine-tuning pretrained models                       ║
║  • Use 10x smaller LR for pretrained layers vs. new head                    ║
║  • Fine-tune for 2-5 epochs max on small datasets                            ║
║                                                                              ║
║  PRODUCTION                                                                  ║
║  • TorchScript / ONNX for deployment (no Python runtime needed)             ║
║  • Quantize (INT8) for 4x speedup on CPU/edge at small accuracy cost        ║
║  • Prune: remove 40-60% of weights with <1% accuracy loss                   ║
║  • Always benchmark on target hardware (CPU, GPU, mobile, edge)             ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""
print(TIPS)

# ── EMA (Exponential Moving Average) ─────────────────────────────────────────
class EMA:
    """EMA of model weights for smoother, better-generalizing parameters."""
    def __init__(self, model, decay=0.999):
        self.model = model
        self.decay = decay
        self.shadow = {k: v.clone().float() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self):
        for k, v in self.model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k] = self.decay * self.shadow[k] + (1-self.decay) * v.float()

    def apply_shadow(self):
        """Apply EMA weights for evaluation."""
        self.backup = {k: v.clone() for k, v in self.model.state_dict().items()}
        self.model.load_state_dict({k: v.to(next(self.model.parameters()).device)
                                     for k, v in self.shadow.items()})

    def restore(self):
        """Restore training weights after evaluation."""
        self.model.load_state_dict(self.backup)

# torch.compile for speedup (PyTorch 2.0+)
# model_compiled = torch.compile(model)  # ~30% speedup, same interface

print("\n✅ Deep Learning Complete Cheat Sheet — All sections done!")
print("🚀 Happy training. May your gradients flow and your losses decrease!")

---
## 📚 Essential Resources

| Resource | URL |
|----------|-----|
| PyTorch Docs | https://pytorch.org/docs |
| TF/Keras Docs | https://tensorflow.org/api_docs |
| HuggingFace | https://huggingface.co/docs |
| Papers With Code | https://paperswithcode.com |
| Andrej Karpathy's lectures | https://youtube.com/@AndrejKarpathy |
| fast.ai | https://course.fast.ai |
| d2l.ai | https://d2l.ai |
| Distill.pub (visualizations) | https://distill.pub |

---
### Key Formulas Quick Reference

```
Attention:      Attn(Q,K,V) = softmax(QKᵀ/√dₖ)·V
Adam update:    θ = θ - α·m̂/(√v̂+ε)  where m̂,v̂ = bias-corrected moments
BCE Loss:       L = -[y·log(p) + (1-y)·log(1-p)]
CE Loss:        L = -Σᵢ yᵢ·log(pᵢ)
KL Divergence:  KL(P||Q) = Σ P(x)·log(P(x)/Q(x))
VAE ELBO:       L = E[log p(x|z)] - KL(q(z|x)||p(z))
Conv out size:  ⌊(W - K + 2P)/S⌋ + 1
Params in FC:   in_features × out_features + out_features (bias)
BatchNorm:      ŷ = γ·(y-μ)/√(σ²+ε) + β
```